In [1]:
import pyscf, ffsim, qiskit
print("all good")

all good


In [2]:
import numpy as np
import pyscf
import pyscf.cc
import ffsim

print("Imports OK")

Imports OK


In [3]:
mol = pyscf.gto.Mole()
mol.atom = "H 0 0 0; H 0 0 0.74"
mol.basis = "sto-3g"
mol.build()

mf = pyscf.scf.RHF(mol)
mf.run()

print("Hartree-Fock energy:", mf.e_tot)

converged SCF energy = -1.11675930739642
Hartree-Fock energy: -1.1167593073964248


In [4]:
cc = pyscf.cc.CCSD(mf)
cc.run()

print("HF energy:  ", mf.e_tot)
print("CCSD energy:", cc.e_tot)
print()
print("t1 shape:", cc.t1.shape)   # (n_occupied, n_virtual)
print("t2 shape:", cc.t2.shape)   # (n_occ, n_occ, n_virt, n_virt)
print()
print("t1 amplitudes:")
print(cc.t1)

E(CCSD) = -1.137283998610438  E_corr = -0.02052469121401287
HF energy:   -1.1167593073964248
CCSD energy: -1.1372839986104377

t1 shape: (1, 1)
t2 shape: (1, 1, 1, 1)

t1 amplitudes:
[[-1.25946929e-16]]


In [5]:
mol_data = ffsim.MolecularData.from_scf(mf)
norb = mol_data.norb
nelec = mol_data.nelec

print("Number of spatial orbitals:", norb)
print("Electrons (alpha, beta):   ", nelec)
print("Qubits needed:             ", 2 * norb)
print("HF energy: ", mol_data.hf_energy)
print("FCI energy:", mol_data.fci_energy)

Number of spatial orbitals: 2
Electrons (alpha, beta):    (1, 1)
Qubits needed:              4
HF energy:  -1.1167593073964248
FCI energy: None


In [6]:
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=cc.t2,
    t1=cc.t1,
    n_reps=1,
)

print("UCJ operator built")

UCJ operator built


In [ ]:
reference = ffsim.hartree_fock_state(norb, nelec)
final_state = ffsim.apply_unitary(reference, ucj_op, norb=norb, nelec=nelec)

ham = ffsim.linear_operator(mol_data.hamiltonian, norb=norb, nelec=nelec)
energy = np.real(np.vdot(final_state, ham @ final_state))

print("LUCJ ansatz energy:", energy)
print("FCI energy:        ", mol_data.fci_energy)
if mol_data.fci_energy is not None:
    print("Error:             ", energy - mol_data.fci_energy)
else:
    print("FCI energy not available")

LUCJ ansatz energy: -1.1321801676192031
FCI energy:         None


TypeError: unsupported operand type(s) for -: 'float' and 'NoneType'